# 02. Бэктест моделей прогноза расходов МО

**Схема.** Rolling-origin с расширяющимся окном: в точке отсечки `T` модель видит только данные до `T`
и прогнозирует месяц `T+h`. Отсечки - с января по ноябрь 2024 г., для `h = 12` возможно одно окно
(декабрь 2023 → декабрь 2024). Горизонты 2, 4, 7 считаются для проверки устойчивости к лагу публикации
таргета: если траты за месяц известны с задержкой в месяц, прогноз «на h» фактически делается на h+1.

Модели, которые оценивались;
- `naive` — траты через h месяцев равны текущим
- `snt` — сезонная наивная с трендом по самому МО: y(T+h−12) · y(T) / y(T−12); нужен год истории МО
- `snt_nat` — сезонная наивная по стране: y(T) · N(T+h−12) / N(T−12), где N — траты по России
  (СберИндекс, с 2018 г), работает с первого месяца ряда МО, в том числе для h = 12
- `prophet` — базовая модель организаторов из условия, без дополнительных параметров
- `prophet_tuned` — усиленный Prophet: годовая сезонность принудительно (порядок 3), мультипликативный режим
- `lgbm_logratio` — глобальная LightGBM, цель log(y(T+h) / y(T)), общая на горизонты 1–7
- `lgbm_resid` — то же, но цель — поправка к национальному сезонному бейзлайну log(y(T+h) / snt_nat)
- `lgbm_resid_nonat` — абляция: `lgbm_resid` без национальных признаков (у них всего ~24 разных значения).

Ансамбли, метрики, интервалы и проверки значимости — в ноутбуке `02b_evaluation`.

In [ ]:
import logging, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import lightgbm as lgb

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load(open(ROOT / "configs" / "backtest.yaml", encoding="utf-8"))
PCFG = yaml.safe_load(open(ROOT / "configs" / "panel.yaml", encoding="utf-8"))
OUT = ROOT / "results"; OUT.mkdir(exist_ok=True)
Y, H, POOL_H = CFG["data"]["target"], CFG["horizons_eval"], CFG["models"]["lightgbm"]["pooled_horizons"]

panel = pd.read_parquet(ROOT / CFG["data"]["panel"])
cal = pd.read_parquet(ROOT / "data" / "processed" / "calendar.parquet").set_index("date")
panel = panel[panel.date >= "2023-01-01"].copy()
yy = panel.pivot(index="mo_id", columns="date", values=Y)

sp = pd.read_parquet(ROOT / "data" / "raw" / PCFG["files"]["sber_spend"]); sp["date"] = pd.to_datetime(sp.period)
NAT = sp[sp.type == "Всего"].set_index("date").value.sort_index()

first, last = pd.Timestamp(CFG["backtest"]["first_origin"]), pd.Timestamp(CFG["backtest"]["last_target"])
def origins(h):
    if h == 12: return [pd.Timestamp(CFG["backtest"]["h12_origin"])]
    return list(pd.date_range(first, last - pd.DateOffset(months=h), freq="MS"))
ORIG = {h: origins(h) for h in H}
M = lambda T, k: T + pd.DateOffset(months=k)
def nat_ratio(T, h): return NAT[M(T, h - 12)] / NAT[M(T, -12)]
print(panel.shape, panel.mo_id.nunique(), "МО")

## 1. Базовые модели

In [ ]:
rows = []
for h in H:
    for T in ORIG[h]:
        tgt = M(T, h)
        if tgt not in yy.columns: continue
        d = pd.DataFrame({"mo_id": yy.index, "origin": T, "h": h, "y_true": yy[tgt].values, "naive": yy[T].values})
        ly, lyT = M(tgt, -12), M(T, -12)
        d["snt"] = yy[ly].values * yy[T].values / yy[lyT].values if (ly in yy.columns and lyT in yy.columns) else np.nan
        d["snt_nat"] = yy[T].values * nat_ratio(T, h)
        rows.append(d)
preds = pd.concat(rows, ignore_index=True)
preds["snt"] = preds.snt.fillna(preds.snt_nat)

## 2. Prophet: по умолчанию и усиленный (выборка МО — стратифицированно по размеру)

In [ ]:
from prophet import Prophet
info = panel.drop_duplicates("mo_id")[["mo_id", "size_class"]]
n = CFG["models"]["prophet"]["sample_n_mo"]
if (OUT / "prophet_sample.csv").exists():
    SAMPLE = pd.read_csv(OUT / "prophet_sample.csv", dtype={"mo_id": str}).mo_id.tolist()
else:
    SAMPLE = (info.groupby("size_class", observed=True, group_keys=False)
              .apply(lambda d: d.sample(max(1, round(n * len(d) / len(info))), random_state=CFG["seed"])).mo_id.tolist())
    pd.Series(SAMPLE, name="mo_id").to_csv(OUT / "prophet_sample.csv", index=False)
SAMPLE = [m for m in SAMPLE if m in yy.index]
cutoffs = sorted({T for h in H for T in ORIG[h]})

def run_prophet(params, cache, mos=None):
    if cache.exists(): return pd.read_parquet(cache)
    rows, t0 = [], time.time()
    for mo in (mos or SAMPLE):
        s = yy.loc[mo].dropna()
        for T in cutoffs:
            hist = s[s.index <= T]
            if len(hist) < 6: continue
            m = Prophet(**{k: v for k, v in params.items() if k != "yearly_order"})
            if params.get("yearly_order"): m.add_seasonality("yearly", period=365.25, fourier_order=params["yearly_order"])
            m.fit(pd.DataFrame({"ds": hist.index, "y": hist.values}))
            fut = pd.DataFrame({"ds": pd.date_range(M(T, 1), periods=12, freq="MS")})
            fc = m.predict(fut)[["ds", "yhat"]]; fc["mo_id"], fc["origin"] = mo, T; rows.append(fc)
    out = pd.concat(rows); out.to_parquet(cache, index=False)
    print(f"{cache.name}: {time.time() - t0:.0f} c"); return out

for name, key in [("prophet", "params"), ("prophet_tuned", "params_tuned")]:
    fc = run_prophet(CFG["models"]["prophet"][key], OUT / f"{name}_forecasts_v2_n{len(SAMPLE)}.parquet")
    fc["h"] = (fc.ds.dt.year - fc.origin.dt.year) * 12 + fc.ds.dt.month - fc.origin.dt.month
    preds = preds.merge(fc[["mo_id", "origin", "h", "yhat"]].rename(columns={"yhat": name}), on=["mo_id", "origin", "h"], how="left")

if CFG["models"]["prophet"].get("all_mo"):
    PC = OUT / "prophet_all_cache"; PC.mkdir(exist_ok=True); allmo = list(yy.index); parts = []
    for i in range(0, len(allmo), 200):
        parts.append(run_prophet(CFG["models"]["prophet"]["params"], PC / f"chunk_{i:05d}.parquet", mos=allmo[i:i + 200]))
    fa = pd.concat(parts); fa["h"] = (fa.ds.dt.year - fa.origin.dt.year) * 12 + fa.ds.dt.month - fa.origin.dt.month
    preds = preds.merge(fa[["mo_id", "origin", "h", "yhat"]].rename(columns={"yhat": "prophet_all"}), on=["mo_id", "origin", "h"], how="left")

## 3. LightGBM: одна модель на горизонты 1–7, обучается один раз на каждую отсечку
Пара (строка t, горизонт h) попадает в обучение, только если месяц t+h уже наступил к отсечке T.
Признаки: всё из ноутбука 01 + горизонт + календарь целевого месяца + национальный сезонный коэффициент.

In [ ]:
drop = {"mo_id", "date", "mo_name_sber", "mo_name_rosstat", "region", "has_target_now"}
FEATS = [c for c in panel.columns if c not in drop and not c.startswith(("target_", "target_date"))]
CATS = [c for c in FEATS if not pd.api.types.is_numeric_dtype(panel[c])]
for c in CATS: panel[c] = panel[c].astype("category")
NAT_FEATS = [c for c in FEATS if c.startswith(("nat_", "sber_"))]
NEWS_FEATS = [c for c in FEATS if c.startswith("news_")]
EXTRA = ["h", "t_month", "t_month_sin", "t_month_cos", "t_workdays", "t_is_december", "log_nat_ratio"]

base = panel.dropna(subset=[Y])
ylong = yy.stack().rename("y_fut")
parts = []
for h in POOL_H:
    d = base.copy(); d["h"] = h; d["target_date"] = d.date + pd.DateOffset(months=h)
    d = d.join(ylong, on=["mo_id", "target_date"])
    d["nat_r"] = d.date.map({T: nat_ratio(T, h) for T in d.date.unique()})
    d["log_nat_ratio"] = np.log(d.nat_r)
    c = cal.loc[d.target_date]
    d["t_month"], d["t_workdays"] = c.month.values, c.workdays.values
    d["t_month_sin"], d["t_month_cos"] = np.sin(2 * np.pi * d.t_month / 12), np.cos(2 * np.pi * d.t_month / 12)
    d["t_is_december"] = (d.t_month == 12).astype(int)
    d["z_logratio"] = np.log(d.y_fut / d[Y])
    d["z_resid"] = np.log(d.y_fut / (d[Y] * d.nat_r))
    parts.append(d)
pairs = pd.concat(parts, ignore_index=True)
print("пар (строка × горизонт):", len(pairs))

LP = CFG["models"]["lightgbm"]["params"]
BASE_FEATS = [f for f in FEATS if f not in NAT_FEATS and f not in NEWS_FEATS]
VARIANTS = {"lgbm_logratio": ("z_logratio", [f for f in FEATS if f not in NEWS_FEATS] + EXTRA, False),
            "lgbm_resid": ("z_resid", [f for f in FEATS if f not in NEWS_FEATS] + EXTRA, True),
            "lgbm_resid_nonat": ("z_resid", BASE_FEATS + EXTRA, True),
            "lgbm_resid_nonat_news": ("z_resid", BASE_FEATS + NEWS_FEATS + EXTRA, True)}
H_LGB = [h for h in H if h in POOL_H]
CACHE = OUT / "lgbm_cache"; CACHE.mkdir(exist_ok=True)
lgb_rows, importances, t0 = [], {}, time.time()
for T in sorted({T for h in H_LGB for T in ORIG[h]}):
    f = CACHE / f"{T.date()}.parquet"
    if f.exists():
        lgb_rows.append(pd.read_parquet(f)); continue
    train = pairs[pairs.target_date <= T]
    test = pairs[(pairs.date == T) & pairs.h.isin(H_LGB)]
    if len(train) < 2000: continue
    res = test[["mo_id", "h"]].assign(origin=T)
    for name, (z, X, is_resid) in VARIANTS.items():
        tr = train[np.isfinite(train[z])]
        cats = [c for c in X if c in CATS]
        model = lgb.LGBMRegressor(**LP, random_state=CFG["seed"]).fit(tr[X], tr[z], categorical_feature=cats)
        ref = test[Y] * (test.nat_r if is_resid else 1.0)
        res[name] = (ref * np.exp(model.predict(test[X]))).values
        imp = pd.Series(model.booster_.feature_importance("gain"), index=X)
        imp.rename("gain").to_frame().to_parquet(CACHE / f"imp_{name}_{T.date()}.parquet")
    res.to_parquet(f, index=False); lgb_rows.append(res)
    print(T.date(), f"{time.time() - t0:.0f} c", flush=True)
preds = preds.merge(pd.concat(lgb_rows), on=["mo_id", "origin", "h"], how="left")

## 4. Сохранение

In [ ]:
preds.to_parquet(OUT / "predictions_all.parquet", index=False)
for name in VARIANTS:
    importance_files = sorted(CACHE.glob(f"imp_{name}_*.parquet"))
    if not importance_files:
        print(f"{name}: в кеше нет важностей признаков; существующая таблица не обновлена")
        continue
    imp = pd.concat([pd.read_parquet(x).gain for x in importance_files], axis=1).mean(axis=1)
    (imp / imp.sum()).sort_values(ascending=False).head(25).round(4).to_csv(OUT / f"feature_importance_{name}.csv")
print(preds.groupby("h").size().to_dict())